# PyLabRobot REPL

A full CPython running in this browser tab, with
[PyLabRobot](https://github.com/PyLabRobot/pylabrobot) importable and its device I/O
rebound to browser APIs. Nothing is installed on your machine and no server runs your
code — the kernel is a Pyodide WebAssembly build in a Web Worker.


## Where your work is saved

Notebooks you create or edit here are saved to **your browser's IndexedDB**, under
three separate keys:

| key | holds |
| --- | --- |
| `praxis-repl-contents` | your notebooks and files |
| `praxis-repl-settings` | editor and UI preferences |
| `praxis-repl-workspaces` | which tabs and panels were open |

What that means in practice:

- **Your notebooks survive a reload, a tab close, and a browser restart.** They are
  real, persistent storage — not session state.
- **They are local to this browser, on this machine, for this origin.** A different
  browser, a different profile, or a different device starts empty. Nothing syncs.
- **Clearing site data deletes them.** "Clear browsing data", a private/incognito
  window closing, or an origin-scoped storage purge takes your notebooks with it.
- **The Python environment itself does not persist.** Packages, imports and variables
  are rebuilt from scratch every time the kernel starts, so re-run the bootstrap cell
  below after each reload or kernel restart.

If a notebook matters, use *File → Download* to get a real `.ipynb` onto your disk.


## 1. Bootstrap

Run this first, once per kernel session. It fetches the Praxis loader, installs the
wheels, and rebinds PyLabRobot's I/O classes to the browser shims (Web Serial,
WebUSB, WebHID, FTDI-over-serial).

It is a fail-closed loader: if anything is missing or has drifted, it raises instead
of leaving you with a half-configured environment. Expect it to take a few seconds.


In [ ]:
import js

# Site root. "/" is correct for the standard build and for a root deploy.
# If you deployed this site under a subpath, set it to that prefix
# (with both leading and trailing slash), e.g. HOST_ROOT = "/praxis/".
HOST_ROOT = "/"

# The loader is fetched and exec'd rather than imported: it lives on the web server
# next to this site, not inside the Pyodide filesystem, and it is the code that makes
# manifest-driven fetching of everything else possible.
xhr = js.XMLHttpRequest.new()
xhr.open("GET", HOST_ROOT + "bootstrap/praxis_bootstrap.py", False)
xhr.send(None)
exec(compile(str(xhr.responseText), "praxis_bootstrap.py", "exec"), globals())

await praxis_main(HOST_ROOT)  # noqa: F821 - defined by the exec above
print("bootstrap complete")


### In your own notebooks, this is two lines

The cell above is written out in full because it is the one place worth
seeing the machinery: the loader lives on the web server next to this site,
so it is fetched and `exec`'d rather than imported.

You do not have to type any of that in a notebook you create. `praxis_boot`
ships in this drive, so it is importable from any notebook here:

```python
import praxis_boot
await praxis_boot.setup()
```

It works out the site root from the kernel worker's own URL, so there is no
`HOST_ROOT` to set and nothing that breaks when the site moves between a root
deploy and a subpath. Run it once per kernel session, same as above.


## 2. Check what you got

`Serial` here is not the desktop `pyserial`-backed class — the bootstrap replaced it
with a Web Serial implementation. The identity check below is the real assertion: it
compares the class *object*, so it cannot pass on a same-named impostor.


In [ ]:
import builtins

import pylabrobot
from pylabrobot.io.serial import Serial

print("PyLabRobot", pylabrobot.__version__)
print("Serial is the browser shim:", Serial is builtins.WebSerial)


## 3. Talking to a real device

Browser device APIs require a **user gesture** — a real click — before they will show
the device picker. Code alone cannot open a device; the browser will refuse. So a
connect step always looks like: your code asks, the page prompts you, you pick a
device.

This also means an unattended notebook cannot connect to hardware. That is a browser
security boundary, not a limitation of this REPL.

Requirements: a Chromium-family browser (Chrome, Edge) over HTTPS or localhost.
Firefox and Safari do not implement Web Serial or WebUSB.


---

Create a new notebook from the file browser on the left to start your own work. This
one is regenerated by the build, so edits to it may be overwritten — save your work
in a notebook of your own.
